# scikit-learn Delaney Regression Baseline

This notebook is the first regression baseline for the project.

Task: predict measured log solubility using the processed Delaney ESOL dataset
Models: Ridge regression and random forest regressor

Why this dataset first:
- small, clean tabular schema
- no missing target values
- interpretable descriptor columns

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
delaney = pd.read_csv(DATA_DIR / 'delaney-processed.csv')

feature_columns = [
    'ESOL predicted log solubility in mols per litre',
    'Minimum Degree',
    'Molecular Weight',
    'Number of H-Bond Donors',
    'Number of Rings',
    'Number of Rotatable Bonds',
    'Polar Surface Area',
]
target_column = 'measured log solubility in mols per litre'

print('Shape:', delaney.shape)
display(delaney.head())

## Data Checks

In [ ]:
display(delaney[feature_columns + [target_column]].describe().T)

plt.figure(figsize=(7, 4))
sns.histplot(delaney[target_column], bins=30, kde=True)
plt.title('Measured Log Solubility Distribution')
plt.show()

In [ ]:
X = delaney[feature_columns]
y = delaney[target_column]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED
)

print('Train:', len(X_train), 'Valid:', len(X_valid), 'Test:', len(X_test))

## Models

In [ ]:
ridge_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0)),
])

forest_model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=RANDOM_SEED,
)

models = {
    'ridge': ridge_model,
    'random_forest': forest_model,
}

results = []
predictions_by_model = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    valid_pred = model.predict(X_valid)
    results.append({
        'model': name,
        'rmse': float(np.sqrt(mean_squared_error(y_valid, valid_pred))),
        'mae': float(mean_absolute_error(y_valid, valid_pred)),
        'r2': float(r2_score(y_valid, valid_pred)),
    })
    predictions_by_model[name] = valid_pred

results_df = pd.DataFrame(results).sort_values('rmse').reset_index(drop=True)
display(results_df.round(4))

In [ ]:
best_model_name = results_df.loc[0, 'model']
best_model = models[best_model_name]
test_pred = best_model.predict(X_test)

test_metrics = pd.DataFrame([
    {
        'model': best_model_name,
        'rmse': float(np.sqrt(mean_squared_error(y_test, test_pred))),
        'mae': float(mean_absolute_error(y_test, test_pred)),
        'r2': float(r2_score(y_test, test_pred)),
    }
])
display(test_metrics.round(4))

In [ ]:
plot_df = pd.DataFrame({
    'true': y_test,
    'predicted': test_pred,
})

plt.figure(figsize=(6, 6))
sns.scatterplot(data=plot_df, x='true', y='predicted', s=40)
line_min = min(plot_df['true'].min(), plot_df['predicted'].min())
line_max = max(plot_df['true'].max(), plot_df['predicted'].max())
plt.plot([line_min, line_max], [line_min, line_max], color='black', linestyle='--')
plt.title(f'Delaney Test Predictions: {best_model_name}')
plt.xlabel('True')
plt.ylabel('Predicted')
plt.show()

## Next Steps

1. add a TensorFlow regressor on the same split
2. compare against PyTorch tabular regression
3. later replace or augment descriptors with RDKit-derived features